<a href="http://landlab.github.io"><img style="float: left; width: 300px;" src="https://landlab.csdms.io/_static/landlab_logo.png"></a>

# 2D Surface Water Flow HLLC Component: Green River Case Study

<hr>
<small>For more Landlab tutorials, click here: <a href="https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html">https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html</a></small>
<hr>

## Overview

This notebook demonstrates the `RiverFlowDynamics_HLLC` component simulating surface water flow over the real terrain of a Green River reach in Utah, USA. Unlike the idealized channel examples, this DEM-based case shows how the HLLC solver handles complex natural topography with irregular bathymetry, dry banks, and spatially variable flow depths.

> **Key numerical differences from `RiverFlowDynamics` (Casulli & Cheng 1992):**
>
> | Feature | RiverFlowDynamics | RiverFlowDynamics_HLLC |
> |---|---|---|
> | Scheme | Semi-implicit, semi-Lagrangian | Explicit Godunov-type (HLLC flux) |
> | Time stepping | User-supplied fixed dt (can exceed CFL) | Adaptive CFL (or user-supplied with warning) |
> | Shock capturing | No | Yes — handles hydraulic jumps and bores |
> | Wet/dry fronts | Via threshold depth | Positive-depth guarantee + hydrostatic reconstruction |
> | Velocity storage | Scalar at links | x- and y-components at nodes |
> | Required input fields | depth, elevation, velocity (link) | depth, topographic elevation only |
>
> The Audusse hydrostatic reconstruction in the HLLC solver ensures **exact well-balancedness** on wet terrain — no spurious velocities from topographic gradients — which is especially valuable over complex natural DEMs.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

# Import the HLLC component
from landlab.components import RiverFlowDynamics_HLLC
from landlab.io import esri_ascii
from landlab.plot.imshow import imshow_grid

## Loading the Digital Elevation Model

We load the DEM of the Green River reach from an ASCII file.

In [ ]:
# Select DEM resolution
# asc_file = "DEM_GreenRiver_100x98_20x20.asc"  # 20 x 20 m resolution
asc_file = "DEM_GreenRiver_80x78_25x25.asc"  # 25 x 25 m resolution

with open(asc_file) as f:
    grid = esri_ascii.load(f, name="topographic__elevation")

teDEM = grid.at_node["topographic__elevation"]

## Define Simulation Parameters

In [ ]:
n = 0.034  # Manning's roughness coefficient, [s/m^(1/3)]

target_time = 1200  # Total simulated time [s]
display_animation_freq = 10  # Redraw every this many simulated seconds

## Set Up Topography

Replace no-data values (-9999) with a high elevation (700 m) so they act as impenetrable walls:

In [ ]:
teDEM = grid.at_node["topographic__elevation"]
grid.at_node["topographic__elevation"] = np.where(teDEM == -9999, 700, teDEM)

### Visualize Initial Topography

In [ ]:
plt.figure(figsize=(8, 6))
imshow_grid(
    grid, "topographic__elevation", cmap="terrain", colorbar_label="Elevation (m)"
)
plt.title("Green River Topography")
plt.show()

## Set Initial Conditions

With `RiverFlowDynamics_HLLC`, only the **depth** field is required. All other output fields (`surface_water__elevation`, `surface_water__x_velocity`, `surface_water__y_velocity`, momentum fields) are created automatically.

In [ ]:
# Only depth field required — HLLC creates all other fields automatically
h = grid.add_zeros("surface_water__depth", at="node")

# Note: Unlike RiverFlowDynamics we do NOT need:
#   vel = grid.add_zeros("surface_water__velocity", at="link")
#   wse = grid.add_zeros("surface_water__elevation", at="node")

## Set Boundary Conditions

Water enters from the **right edge** and flows **left** (negative x-direction). `RiverFlowDynamics_HLLC` uses **node-centered u/v velocity components** instead of link-based scalar velocities.

The x-velocity at entry nodes is **negative** (leftward). We compute it from discharge conservation: $u = -Q / (\Delta x \cdot \sum N_{wet})$ distributed as $u_i = u_\text{mean} / h_i$, matching the original depth-weighted approach.

In [ ]:
# Entry nodes at the right edge
fixed_entry_nodes = grid.nodes_at_right_edge

# Water-surface elevation from thalweg + depth
thalweg = grid.at_node["topographic__elevation"][fixed_entry_nodes].min()
WSE = thalweg + 5.4559  # target WSE

# Depth at each entry node (clip negatives to 0)
entry_nodes_h_values = WSE - grid.at_node["topographic__elevation"][fixed_entry_nodes]
entry_nodes_h_values = np.where(entry_nodes_h_values < 0, 0, entry_nodes_h_values)

# x-velocity at entry nodes — negative because flow is rightward-to-leftward
# Discharge Q = 247 m³/s distributed uniformly over the wet cross-section
wet_mask = entry_nodes_h_values > 0
n_wet = np.sum(wet_mask)
safe_h = np.where(wet_mask, entry_nodes_h_values, 1.0)

# Depth-weighted velocity: u_i = -Q / (dx * sum_h), proportional 1/h_i for constant discharge
# Equivalent to: each wet node carries Q/(dx * n_wet) discharge, so u = q/h = Q/(dx*n_wet*h)
entry_nodes_u_values = np.where(
    wet_mask, -(247.0 / (grid.dx * n_wet)) * (1.0 / safe_h), 0.0
)
entry_nodes_v_values = np.zeros(len(fixed_entry_nodes))  # no cross-flow

### Visualize Entry Cross-section

In [ ]:
# Entry cross-section data
y_profile = grid.y_of_node[fixed_entry_nodes]
z_profile = grid.at_node["topographic__elevation"][fixed_entry_nodes]
wse_profile = z_profile + entry_nodes_h_values
h_profile = entry_nodes_h_values

# Sort by y for clean plotting
sort_idx = np.argsort(y_profile)
y_profile = y_profile[sort_idx]
z_profile = z_profile[sort_idx]
wse_profile = wse_profile[sort_idx]
h_profile = h_profile[sort_idx]

# Keep only truly wet nodes
h_tol = 0.05  # adjust if needed
wet_mask = h_profile > h_tol

# Plot limits
fixed_x_min = np.min(y_profile)
fixed_x_max = np.max(y_profile)
fixed_y_min = np.min(z_profile) - 2.0
fixed_y_max = np.max(z_profile) + 2.0

fig, ax = plt.subplots(figsize=(7.2, 4.4))

# Channel bed
ax.plot(
    y_profile,
    z_profile,
    color="saddlebrown",
    linewidth=2.5,
    label="Channel Bed",
    zorder=3,
)
ax.fill_between(
    y_profile,
    fixed_y_min,
    z_profile,
    color="saddlebrown",
    alpha=0.4,
    zorder=1,
)

# Water only where wetted
if np.any(wet_mask):
    y_wet = y_profile[wet_mask]
    z_wet = z_profile[wet_mask]
    wse_wet = wse_profile[wet_mask]

    ax.plot(
        y_wet,
        wse_wet,
        color="deepskyblue",
        linewidth=2.5,
        label="Water Surface",
        zorder=4,
    )
    ax.fill_between(
        y_wet,
        z_wet,
        wse_wet,
        color="deepskyblue",
        alpha=0.5,
        zorder=2,
    )

ax.set_xlim(fixed_x_min, fixed_x_max)
ax.set_ylim(fixed_y_min, fixed_y_max)
ax.set_title(
    "Entry Cross-section of Green River",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("Distance across cross-section [m]", fontsize=11)
ax.set_ylabel("Elevation [m]", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.6, zorder=0)
ax.legend(loc="upper right", framealpha=1.0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## Initialize the RiverFlowDynamics_HLLC Component

Key differences from `RiverFlowDynamics`:
- **No `dt` at construction** — adaptive CFL stepping by default.
- **`entry_nodes_u_values` / `entry_nodes_v_values`** replace `entry_links_vel_values`.
- **No `fixed_entry_links`** — the HLLC scheme is node-based.
- All boundaries default to **transmissive (zero-gradient) outflow**.

In [ ]:
rfd = RiverFlowDynamics_HLLC(
    grid,
    mannings_n=n,
    fixed_entry_nodes=fixed_entry_nodes,
    entry_nodes_h_values=entry_nodes_h_values,
    entry_nodes_u_values=entry_nodes_u_values,
    entry_nodes_v_values=entry_nodes_v_values,
    # All edges transmissive by default; add wall_edges={"top","bottom"} if channel walls needed
)

## Run the Simulation

We run for 1200 seconds. The CFL criterion dynamically limits each step based on the current wave speed, automatically handling the highly variable depths in a natural channel.

In [ ]:
# -------------------------------------------------
# Pre-calculate geometry and fixed plotting limits
# -------------------------------------------------
nrows, ncols = grid.shape
X = grid.x_of_node.reshape((nrows, ncols))
Y = grid.y_of_node.reshape((nrows, ncols))
z0 = grid.at_node["topographic__elevation"].reshape((nrows, ncols))
topo_vmin = np.min(z0)
topo_vmax = np.max(z0)
depth_vmin = 0.0
depth_vmax = max(0.5, float(np.max(grid.at_node["surface_water__depth"])))
h_tol = 1e-6


# -------------------------------------------------
# Styled display function
# -------------------------------------------------
def update_display(step, elapsed, target_time, t0, show_progress=True):
    clear_output(wait=True)
    z = grid.at_node["topographic__elevation"].reshape((nrows, ncols))
    h = grid.at_node["surface_water__depth"].reshape((nrows, ncols))
    fig, ax = plt.subplots(figsize=(7.2, 5.8))
    ax.pcolormesh(X, Y, z, cmap="Greys", shading="auto", vmin=topo_vmin, vmax=topo_vmax)
    h_masked = np.ma.masked_where(h <= h_tol, h)
    water_plot = ax.pcolormesh(
        X,
        Y,
        h_masked,
        cmap="Blues",
        shading="auto",
        vmin=depth_vmin,
        vmax=depth_vmax,
        alpha=0.85,
    )
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(
        f"Topography and Water Depth - Step {step}  [t = {elapsed:.2f} s]",
        fontsize=13,
        fontweight="bold",
    )
    ax.set_xlabel("x [m]", fontsize=11)
    ax.set_ylabel("y [m]", fontsize=11)
    cbar = fig.colorbar(water_plot, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Water Depth [m]", fontsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()
    if show_progress:
        wall = time.time() - t0
        frac = elapsed / target_time if target_time > 0 else 0.0
        eta = wall * (1 / frac - 1) if frac > 0 else float("nan")
        print(
            f"Step {step}  ({frac:.1%})  "
            f"sim-time {elapsed:.2f} s / {target_time:.1f} s  "
            f"dt {rfd.current_dt:.4f} s  wall {wall:.1f} s  ETA {eta:.1f} s"
        )


# -------------------------------------------------
# Animation loop
# -------------------------------------------------
t0 = time.time()
step = 0
last_display_time = 0.0

update_display(step, 0.0, target_time, t0, show_progress=False)

while rfd.elapsed_time < target_time:
    rfd.run_one_step()
    step += 1
    if rfd.elapsed_time >= last_display_time + display_animation_freq:
        update_display(step, rfd.elapsed_time, target_time, t0, show_progress=True)
        last_display_time = rfd.elapsed_time

# Final frame (in case the last step didn't land on a display interval)
if rfd.elapsed_time > last_display_time:
    update_display(step, rfd.elapsed_time, target_time, t0, show_progress=True)

print("Done.")

## Visualize Flow Velocity Magnitude

With `RiverFlowDynamics_HLLC`, velocity is stored as **x- and y-components at nodes**. Magnitude is $|\mathbf{u}| = \sqrt{u^2 + v^2}$ computed directly — no link interpolation required:

In [ ]:
u = grid.at_node["surface_water__x_velocity"]
v = grid.at_node["surface_water__y_velocity"]
velocity_magnitude = np.sqrt(u**2 + v**2)
grid.at_node["velocity_magnitude"] = velocity_magnitude

nrows, ncols = grid.shape
X = grid.x_of_node.reshape((nrows, ncols))
Y = grid.y_of_node.reshape((nrows, ncols))
Z = grid.at_node["topographic__elevation"].reshape((nrows, ncols))
H = grid.at_node["surface_water__depth"].reshape((nrows, ncols))
Vmag = velocity_magnitude.reshape((nrows, ncols))

h_tol = 1e-6
Vmag_masked = np.ma.masked_where(H <= h_tol, Vmag)

topo_vmin = np.min(Z)
topo_vmax = np.max(Z)
vel_vmin = 0.0
vel_vmax = max(0.5, float(np.nanmax(Vmag)))

fig, ax = plt.subplots(figsize=(7.4, 5.8))
ax.pcolormesh(X, Y, Z, cmap="Greys", shading="auto", vmin=topo_vmin, vmax=topo_vmax)
vel_plot = ax.pcolormesh(
    X,
    Y,
    Vmag_masked,
    cmap="plasma",
    shading="auto",
    vmin=vel_vmin,
    vmax=vel_vmax,
    alpha=0.90,
)
ax.set_aspect("equal", adjustable="box")
ax.set_title(
    f"Topography and Flow Velocity Magnitude  [t = {rfd.elapsed_time:.2f} s]",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("x [m]", fontsize=11)
ax.set_ylabel("y [m]", fontsize=11)
cbar = fig.colorbar(vel_plot, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Velocity Magnitude [m/s]", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## Physics of River Flow in Natural Channels

This simulation demonstrates several key physical processes:

1. **Channel Hydraulics**: The balance between gravity and friction governs flow. Water accelerates in steeper sections and decelerates in flatter areas.
2. **Complex Flow Patterns**: Irregular natural channel geometry creates 2D velocity patterns — `surface_water__x_velocity` and `surface_water__y_velocity` at nodes capture both along-channel and cross-channel components simultaneously.
3. **Conservation Principles**: The finite-volume HLLC discretization exactly conserves mass (water volume) and very nearly conserves momentum across the domain.
4. **Well-balanced treatment of topography**: Audusse hydrostatic reconstruction ensures that a lake at rest remains exactly at rest — no spurious currents arise from the irregular DEM.
5. **Adaptive CFL stepping**: The CFL criterion automatically reduces time step size in regions of fast flow (deep thalweg, constrictions) and allows larger steps in shallow or slow regions.

## Conclusion

We demonstrated `RiverFlowDynamics_HLLC` routing the Green River over real terrain. Key insights:

1. **Realistic Flow Patterns**: The HLLC solver captures natural flow concentration in topographic lows.
2. **Inflow/Outflow Balance**: Momentum (`hu`) stored at nodes makes discharge calculation straightforward.
3. **Well-balanced DEM Routing**: Hydrostatic reconstruction prevents numerical artifacts on the irregular DEM.
4. **Velocity Fields**: Node-centered x/y velocity components provide a complete 2D picture of the flow field.

-- --
### And that's it!

Nice work completing this tutorial. You now know how to use `RiverFlowDynamics_HLLC` for simulations with real topographic data :)

-- --

### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>